# Task 4: Context-Aware Chatbot Using RAG
**DevelopersHub Corporation — AI/ML Engineering Internship**

---

## Problem Statement
A standard LLM only knows what it was trained on — it can't answer questions about your specific documents or recent data. **Retrieval-Augmented Generation (RAG)** solves this by fetching relevant documents from a knowledge base and feeding them as context to the LLM before generating a response.

## Objective
- Build a RAG pipeline using LangChain
- Embed documents into a FAISS vector store
- Retrieve relevant chunks based on user queries
- Maintain conversation memory across turns
- Deploy as an interactive Streamlit chatbot

## How RAG Works
```
User Query
    ↓
Embed query → Search FAISS vector store → Retrieve top-k relevant chunks
    ↓
Combine chunks + chat history + query → Send to LLM
    ↓
LLM generates grounded response
```

---

## Section 1 — Installations & Imports

In [ ]:
# Uncomment to install in Colab
# !pip install langchain langchain-community langchain-groq \
#              sentence-transformers faiss-cpu python-dotenv \
#              pypdf tiktoken -q

: 

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

# LangChain core
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# LLM — Groq (free, fast)
from langchain_groq import ChatGroq

# Environment
from dotenv import load_dotenv
load_dotenv()

print('All libraries imported successfully.')

: 

In [ ]:
# ── Set your Groq API key ─────────────────────────────────────
# Get a FREE key at: https://console.groq.com
# Either set it in .env file or paste directly here

GROQ_API_KEY = os.getenv('GROQ_API_KEY') or 'your_groq_api_key_here'
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

print('API key configured.')

---
## Section 2 — Dataset Loading & Preprocessing

We use a custom AI/ML knowledge base (`knowledge_base.txt`). The same pipeline works with any text files, PDFs, or Wikipedia pages — just swap the loader.

In [ ]:
# ── Load knowledge base ───────────────────────────────────────
with open('knowledge_base.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()

print(f'Knowledge base loaded.')
print(f'Total characters : {len(raw_text):,}')
print(f'Total paragraphs : {len(raw_text.split(chr(10)+chr(10)))}')
print(f'\nPreview (first 300 chars):')
print(raw_text[:300])

In [ ]:
# ── Wrap in LangChain Document format ────────────────────────
documents = [Document(page_content=raw_text, metadata={'source': 'knowledge_base.txt'})]

# ── Split into chunks ─────────────────────────────────────────
# chunk_size=500: each chunk holds ~500 characters
# chunk_overlap=50: 50 characters shared between adjacent chunks
# Overlap prevents answers from being cut off at chunk boundaries

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', '']
)

chunks = splitter.split_documents(documents)

print(f'Total chunks created : {len(chunks)}')
print(f'\nExample chunk:')
print(f'  Content : {chunks[0].page_content[:200]}...')
print(f'  Metadata: {chunks[0].metadata}')

---
## Section 3 — Document Embedding & Vector Store

We convert each chunk into a dense vector using `sentence-transformers`. Semantically similar chunks produce similar vectors, enabling similarity search.

FAISS indexes these vectors for fast nearest-neighbour retrieval.

In [ ]:
# ── Load embedding model ──────────────────────────────────────
# all-MiniLM-L6-v2 is fast, lightweight, and excellent for RAG
# Downloads ~80MB once and caches locally

print('Loading embedding model (downloads once, ~80MB)...')

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

print('Embedding model loaded.')

# Show what an embedding looks like
sample_embedding = embeddings.embed_query('What is machine learning?')
print(f'Embedding dimensions : {len(sample_embedding)}')
print(f'First 5 values       : {sample_embedding[:5]}')

In [ ]:
# ── Build FAISS vector store ──────────────────────────────────
print('Building FAISS vector store...')

vectorstore = FAISS.from_documents(chunks, embeddings)

# Save locally so we don't rebuild every time
vectorstore.save_local('faiss_index')

print(f'Vector store built and saved to ./faiss_index/')
print(f'Total vectors indexed: {vectorstore.index.ntotal}')

In [ ]:
# ── Test retrieval ────────────────────────────────────────────
# Verify the vector store returns relevant chunks for a query

retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}      # return top 3 most relevant chunks
)

test_query  = 'What is RAG and how does it work?'
test_results = retriever.invoke(test_query)

print(f'Query: "{test_query}"')
print(f'\nTop {len(test_results)} retrieved chunks:\n')
for i, doc in enumerate(test_results):
    print(f'  Chunk {i+1}: {doc.page_content[:150]}...')
    print()

---
## Section 4 — RAG Chain with Conversation Memory

We wire everything together with LangChain's `ConversationalRetrievalChain`:
- **Retriever** → fetches relevant chunks per query
- **Memory** → stores chat history so the bot remembers previous turns
- **LLM** → generates responses grounded in retrieved context

In [ ]:
# ── Initialise LLM (Groq — free & fast) ──────────────────────
# Get free API key at https://console.groq.com
# llama-3.1-8b-instant is fast and capable for RAG tasks

llm = ChatGroq(
    model='llama-3.1-8b-instant',
    temperature=0.2,         # low temperature = more factual, less creative
    max_tokens=512,
    api_key=GROQ_API_KEY
)

print('LLM initialised: llama-3.1-8b-instant via Groq')

In [ ]:
# ── Conversation Memory ───────────────────────────────────────
# Stores the full chat history
# return_messages=True keeps it as message objects (required by chain)

memory = ConversationBufferMemory(
    memory_key='chat_history',
    return_messages=True,
    output_key='answer'
)

print('Memory initialised.')

In [ ]:
# ── Build ConversationalRetrievalChain ────────────────────────
# This chain:
#   1. Takes the user query + chat history
#   2. Retrieves relevant chunks from FAISS
#   3. Passes everything to the LLM
#   4. Returns a grounded answer and updates memory

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    verbose=False
)

print('RAG chain ready.')
print('Pipeline: Query → FAISS Retrieval → LLM → Response')

---
## Section 5 — Test the Chatbot

We run a multi-turn conversation to verify:
1. Answers are grounded in the knowledge base
2. The bot remembers previous turns (context memory)

In [ ]:
# ── Helper function ───────────────────────────────────────────
def chat(question: str):
    """Send a question to the RAG chain and print the response."""
    result = qa_chain.invoke({'question': question})
    answer = result['answer']
    sources = result.get('source_documents', [])

    print(f'You  : {question}')
    print(f'Bot  : {answer}')
    if sources:
        print(f'Source: {sources[0].page_content[:100]}...')
    print('-' * 60)
    return answer

In [ ]:
# ── Turn 1: Factual question ──────────────────────────────────
chat('What is Retrieval-Augmented Generation?')

In [ ]:
# ── Turn 2: Follow-up using context memory ────────────────────
# The bot should remember we were talking about RAG
chat('What role does FAISS play in it?')

In [ ]:
# ── Turn 3: Different topic ───────────────────────────────────
chat('Explain the difference between deep learning and machine learning.')

In [ ]:
# ── Turn 4: Reference previous answer ────────────────────────
# Tests memory — bot should recall what it said about transformers
chat('What architecture are modern LLMs based on?')

In [ ]:
# ── View full conversation history ────────────────────────────
print('Full conversation history in memory:')
print(f'Total turns : {len(memory.chat_memory.messages) // 2}\n')
for i, msg in enumerate(memory.chat_memory.messages):
    role = 'Human' if i % 2 == 0 else 'Bot'
    print(f'  [{role}]: {msg.content[:100]}...')

---
## Section 6 — Final Summary & Insights

In [ ]:
print('=' * 58)
print('          TASK 4 — FINAL SUMMARY')
print('=' * 58)
print(f"""
Pipeline Components:
─────────────────────────────────────────────────────
  Knowledge Base   : knowledge_base.txt (custom corpus)
  Text Splitter    : RecursiveCharacterTextSplitter
                     chunk_size=500, overlap=50
  Embedding Model  : sentence-transformers/all-MiniLM-L6-v2
                     384-dimensional vectors
  Vector Store     : FAISS (saved to ./faiss_index/)
  Retrieval        : Top-3 similarity search
  LLM              : llama-3.1-8b-instant via Groq API
  Memory           : ConversationBufferMemory
  Framework        : LangChain ConversationalRetrievalChain

Key Insights:
─────────────────────────────────────────────────────
1. RAG grounds LLM responses in real documents,
   reducing hallucinations significantly.

2. Conversation memory allows multi-turn dialogue —
   the bot understands follow-up questions.

3. The same pipeline works with any corpus —
   swap knowledge_base.txt with PDFs, websites,
   or internal documents.

4. FAISS enables sub-millisecond retrieval even
   on large document collections.

Deploy: streamlit run app.py
""")